# Geometry-MMALS G1 v1.1.3
## Functional Memory Transport under Matched Remembered-Source Access

**Status:** corrected C0 specification and executable Colab; gradient-scale calibration required before the five-seed pilot.  
**Build revision:** `functional-memory-transport-c0-r2`.

v1.1.3 fixes one linear router and varies only the memory objective. It is the final planned experiment in the fully frozen-host regime.

Treatments:
- M0 current only;
- M1 limited replay CE;
- M2 nominal route anchoring;
- M3 functional transport anchoring under the common host cost matrix;
- M4 joint functional memory (functional transport + latent + output preservation).

Run order:
1. `G1_PROFILE=development_calibration`;
2. commit the proposed `lambda_F` and calibration SHA-256 into the YAML;
3. set calibration status to `locked` and `pilot_lock: true`;
4. run `G1_PROFILE=pilot`.


## 0. Release-integrity setup

The notebook expects package version 1.1.3 and never patches source at runtime.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, sys, shutil, importlib, subprocess
from pathlib import Path

EXPECTED_PACKAGE_VERSION = "1.1.3"
EXPECTED_BUILD_REVISION = "functional-memory-transport-c0-r2"
REPO_URL = "https://github.com/gharbonnier78/geometry-mmalls-g1.git"
REPO_DIR = Path("/content/geometry-mmalls-g1")
SRC_DIR = REPO_DIR / "src"
FORCE_REFRESH = False

if FORCE_REFRESH and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-cache-dir", "--no-deps", "-e", str(REPO_DIR)], check=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
for name in list(sys.modules):
    if name == "geometry_mmalls" or name.startswith("geometry_mmalls."):
        del sys.modules[name]
importlib.invalidate_caches()
import geometry_mmalls
loaded_from = Path(geometry_mmalls.__file__).resolve()
assert geometry_mmalls.__version__ == EXPECTED_PACKAGE_VERSION
assert loaded_from.parent == (SRC_DIR / "geometry_mmalls").resolve()
print("Geometry-MMALS", geometry_mmalls.__version__, "from", loaded_from)


## 1. Configuration and preregistered memory treatments

Only a linear router is trained. The sensory encoder, context encoder, host bank, synthesis normalization, classifier, and common functional cost matrix are frozen and shared.


In [ ]:
from pathlib import Path
import copy, hashlib, json, math, random, time, platform
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import yaml
from scipy import stats
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import MNIST

from geometry_mmalls.data import FixedAngleMNIST, MultiAngleMNIST
from geometry_mmalls.geometry import (
    paired_route_geometry_loss_stationary,
    paired_context_geometry_loss,
    context_path_spread_loss,
    cross_source_fiber_alignment_loss,
    factor_centroid_geometry_loss,
)
from geometry_mmalls.model import GeometryMMALS, ResidualHost, SmallConvEncoder
from geometry_mmalls.functional_routing import (
    FunctionalRoutingMMALS,
    LinearContextRouter,
    UniformRouter,
    host_functional_diversity_loss,
    host_output_cost_matrix,
    parameter_count,
    route_weighted_synthesis,
    usage_balance_loss,
    entropic_transport_cost,
)
from geometry_mmalls.memory_transport import (
    compile_root_gaussian,
    distributional_functional_transport,
    distillation_kl,
    functional_path_statistics,
    functional_transport_anchor_loss,
    isotropic_root_gaussian,
    latent_anchor_loss,
    root_gaussian_nll,
    root_route_anchor_loss,
)

NOTEBOOK_VERSION = "1.1.3"
BUILD_REVISION = "functional-memory-transport-c0-r2"
base_config = yaml.safe_load(Path("configs/rotated_mnist_g1.yaml").read_text())
protocol = yaml.safe_load(Path("configs/rotated_mnist_g1_v113.yaml").read_text())
assert protocol["build_revision"] == BUILD_REVISION

RUN_PROFILE = os.environ.get("G1_PROFILE", "development_calibration")
CALIBRATION_ONLY = RUN_PROFILE == "development_calibration"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPLIT_SEED = int(protocol["experiment"]["fixed_split_seed"])
SENSORY_SEED = int(protocol["experiment"]["fixed_sensory_seed"])
CALIBRATION_CFG = protocol["gradient_scale_calibration"]
RECONSTRUCTION_CFG = protocol["reconstruction_acceptance"]

if RUN_PROFILE in {"development_calibration", "development"}:
    MODEL_SEEDS = [0]
    TRAIN_SOURCE_LIMIT, TEST_SOURCE_LIMIT = 256, 320
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 1500, 750
    SENSORY_EPOCHS = CONTEXT_STAGE_EPOCHS = BANK_STAGE_EPOCHS = ROUTER_STAGE_EPOCHS = 1
    SOURCE_BATCH_SIZE, SOURCE_BOOTSTRAP_SAMPLES = 64, 300
elif RUN_PROFILE == "pilot":
    MODEL_SEEDS = list(map(int, protocol["experiment"]["pilot_model_seeds"]))
    TRAIN_SOURCE_LIMIT, TEST_SOURCE_LIMIT = 512, 640
    SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 2000, 1000
    SENSORY_EPOCHS = CONTEXT_STAGE_EPOCHS = BANK_STAGE_EPOCHS = ROUTER_STAGE_EPOCHS = 2
    SOURCE_BATCH_SIZE = 64
    SOURCE_BOOTSTRAP_SAMPLES = int(protocol["statistics"]["source_bootstrap_samples"])
elif RUN_PROFILE == "qualification":
    MODEL_SEEDS = list(map(int, protocol["experiment"]["qualification_model_seeds"]))
    TRAIN_SOURCE_LIMIT = int(base_config["paired_protocol"]["full_source_limit"])
    TEST_SOURCE_LIMIT, SENSORY_SOURCE_LIMIT, SENSORY_TEST_LIMIT = 2500, 12000, 5000
    SENSORY_EPOCHS = int(base_config["training"]["sensory_pretrain_epochs"])
    CONTEXT_STAGE_EPOCHS = BANK_STAGE_EPOCHS = ROUTER_STAGE_EPOCHS = int(base_config["training"]["stage_epochs"])
    SOURCE_BATCH_SIZE = int(base_config["paired_protocol"]["source_batch_size"])
    SOURCE_BOOTSTRAP_SAMPLES = int(protocol["statistics"]["source_bootstrap_samples"])
else:
    raise ValueError("G1_PROFILE must be development_calibration, development, pilot, or qualification")

if not CALIBRATION_ONLY:
    locked = (
        CALIBRATION_CFG.get("status") == "locked"
        and bool(CALIBRATION_CFG.get("pilot_lock"))
        and isinstance(CALIBRATION_CFG.get("calibrated_functional_transport_weight"), (int, float))
        and isinstance(CALIBRATION_CFG.get("calibration_report_sha256"), str)
        and len(CALIBRATION_CFG.get("calibration_report_sha256")) == 64
    )
    if RUN_PROFILE in {"pilot", "qualification"} and not locked:
        raise RuntimeError(
            "Pilot blocked: run development_calibration, commit lambda_F and the calibration report SHA-256, "
            "then set gradient_scale_calibration.status=locked and pilot_lock=true."
        )

TRAIN_ANGLES = list(map(float, base_config["data"]["train_angles"]))
CURRICULUM = list(map(float, protocol["experiment"]["curriculum"]))
DENSE_EVAL_ANGLES = list(map(float, protocol["experiment"]["dense_eval_angles"]))
HELDOUT_ANGLES = [a for a in DENSE_EVAL_ANGLES if a not in TRAIN_ANGLES]
CONTEXT_DIM = 4
NUM_HOSTS = int(protocol["experiment"]["host_count"])
MEMORY_PER_ANGLE = int(protocol["experiment"]["memory_sources_per_angle"])
DISTRIBUTIONAL_LIMIT = int(protocol["experiment"]["distributional_transport_sources"])
TAU = float(protocol["router"]["temperature"])
METHOD_SPECS = copy.deepcopy(protocol["memory_treatments"])
FUNCTIONAL_WEIGHT = CALIBRATION_CFG.get("calibrated_functional_transport_weight")
if FUNCTIONAL_WEIGHT is not None:
    METHOD_SPECS["m3_functional_transport"]["functional_transport"] = float(FUNCTIONAL_WEIGHT)
    METHOD_SPECS["m4_joint_functional_memory"]["functional_transport"] = float(FUNCTIONAL_WEIGHT)
METHODS = list(METHOD_SPECS)
print({
    "version": NOTEBOOK_VERSION,
    "revision": BUILD_REVISION,
    "profile": RUN_PROFILE,
    "device": str(DEVICE),
    "seeds": MODEL_SEEDS,
    "methods": METHODS,
    "calibration_status": CALIBRATION_CFG.get("status"),
    "lambda_R": CALIBRATION_CFG.get("route_anchor_weight"),
    "lambda_F": FUNCTIONAL_WEIGHT,
})


## 2. Numerical self-tests

These tests cover paired route anchoring, functional transport, path statistics, compiled root-Gaussian memory, and distillation.


In [ ]:
torch.manual_seed(113)
probe_routes = torch.softmax(torch.randn(16, NUM_HOSTS), dim=-1)
probe_routes_2 = torch.softmax(torch.randn(16, NUM_HOSTS), dim=-1)
probe_cost = torch.rand(NUM_HOSTS, NUM_HOSTS)
probe_cost = 0.5 * (probe_cost + probe_cost.T)
probe_cost.fill_diagonal_(0.0)
assert torch.isfinite(root_route_anchor_loss(probe_routes, probe_routes_2))
assert torch.isfinite(functional_transport_anchor_loss(probe_routes, probe_routes_2, probe_cost))
assert torch.isfinite(distributional_functional_transport(
    probe_routes[:8], probe_routes_2[:8], probe_cost,
    route_iterations=25, cloud_iterations=25,
))
stats_probe = functional_path_statistics([probe_routes, probe_routes_2, probe_routes], probe_cost, iterations=25)
assert stats_probe.cumulative_length >= 0.0
assert stats_probe.endpoint_distance >= 0.0
assert math.isfinite(stats_probe.path_to_endpoint_cost_ratio)
memory_probe = compile_root_gaussian(probe_routes)
isotropic_probe = isotropic_root_gaussian(memory_probe)
assert torch.isfinite(root_gaussian_nll(probe_routes_2, memory_probe))
assert torch.isfinite(root_gaussian_nll(probe_routes_2, isotropic_probe))
assert torch.isfinite(distillation_kl(torch.randn(8,10), torch.randn(8,10)))
print("Memory-transport numerical gates: PASS")


## 3. Source-disjoint partitions and limited-memory split

The memory buffer is a fixed subset of training identities. Evaluation partitions are disjoint from both current training and memory fitting.


In [ ]:
root = Path(base_config["data"]["root"])
base_train = MNIST(root=str(root), train=True, download=True)
base_test = MNIST(root=str(root), train=False, download=True)
rng = np.random.default_rng(SPLIT_SEED)
train_perm = rng.permutation(len(base_train))
test_perm = rng.permutation(len(base_test))

sensory_indices = train_perm[:SENSORY_SOURCE_LIMIT].tolist()
train_indices = train_perm[:TRAIN_SOURCE_LIMIT].tolist()
memory_indices = train_indices[:MEMORY_PER_ANGLE]
current_indices = train_indices[MEMORY_PER_ANGLE:]
test_indices = test_perm[:TEST_SOURCE_LIMIT].tolist()
sensory_test_indices = test_perm[:SENSORY_TEST_LIMIT].tolist()
parts = np.array_split(np.asarray(test_indices), 6)
metric_indices, memory_eval_indices, path_eval_indices, compiled_fit_indices, compiled_eval_indices, safety_eval_indices = [p.tolist() for p in parts]

def checksum(values):
    return hashlib.sha256(",".join(map(str, values)).encode()).hexdigest()

def generator(seed):
    g = torch.Generator(); g.manual_seed(int(seed)); return g

def fixed_loader(angle, train, indices, shuffle, loader_seed=0, batch_size=128):
    ds = Subset(FixedAngleMNIST(root, angle=angle, train=train, download=True), indices)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, generator=generator(loader_seed) if shuffle else None)

def multi_loader(angles, train, indices, shuffle, loader_seed=0, batch_size=None):
    ds = MultiAngleMNIST(root, angles=angles, train=train, indices=indices, download=True)
    return DataLoader(ds, batch_size=batch_size or SOURCE_BATCH_SIZE, shuffle=shuffle, num_workers=0, generator=generator(loader_seed) if shuffle else None)

memory_split_manifest = {
    "split_seed": SPLIT_SEED,
    "budget_definition": protocol["experiment"]["memory_budget_definition"],
    "current_train_sha256": checksum(current_indices),
    "memory_train_sha256": checksum(memory_indices),
    "test_sha256": checksum(test_indices),
    "metric_sha256": checksum(metric_indices),
    "memory_eval_sha256": checksum(memory_eval_indices),
    "path_eval_sha256": checksum(path_eval_indices),
    "compiled_fit_sha256": checksum(compiled_fit_indices),
    "compiled_eval_sha256": checksum(compiled_eval_indices),
    "safety_eval_sha256": checksum(safety_eval_indices),
    "memory_sources_per_angle": len(memory_indices),
    "selection_rule": "fixed split permutation; first memory_sources_per_angle indices",
}
assert set(current_indices).isdisjoint(memory_indices)
print(memory_split_manifest)


## 4. Reproduce and freeze the aligned context checkpoint


In [ ]:
random.seed(SENSORY_SEED); np.random.seed(SENSORY_SEED); torch.manual_seed(SENSORY_SEED)
latent_dim = int(base_config["model"]["latent_dim"])
encoder = SmallConvEncoder(latent_dim).to(DEVICE)
sensory_head = nn.Linear(latent_dim, 10).to(DEVICE)
sensory_optimizer = torch.optim.AdamW(list(encoder.parameters()) + list(sensory_head.parameters()), lr=float(base_config["training"]["learning_rate"]), weight_decay=float(base_config["training"]["weight_decay"]))
sensory_history = []
for epoch in range(SENSORY_EPOCHS):
    total = correct = 0; loss_sum = 0.0
    for x, y, _, _ in fixed_loader(0.0, True, sensory_indices, True, SENSORY_SEED * 1000 + epoch):
        x, y = x.to(DEVICE), y.to(DEVICE)
        logits = sensory_head(encoder(x)); loss = F.cross_entropy(logits, y)
        sensory_optimizer.zero_grad(set_to_none=True); loss.backward(); sensory_optimizer.step()
        total += y.numel(); correct += int((logits.argmax(1) == y).sum()); loss_sum += float(loss.detach()) * y.numel()
    sensory_history.append({"epoch": epoch, "loss": loss_sum/total, "accuracy": correct/total})
sensory_state = copy.deepcopy(encoder.state_dict())

def state_hash(state_dict):
    h = hashlib.sha256()
    for name in sorted(state_dict):
        tensor = state_dict[name].detach().cpu().contiguous()
        h.update(name.encode()); h.update(tensor.numpy().tobytes())
    return h.hexdigest()

def epoch_seed(model_seed, stage, epoch, offset=0):
    return int(model_seed) * 1_000_000 + stage * 10_000 + epoch + int(offset)

def build_context_source_model():
    enc = SmallConvEncoder(latent_dim).to(DEVICE); enc.load_state_dict(sensory_state)
    return GeometryMMALS(enc, latent_dim=latent_dim, context_dim=CONTEXT_DIM, num_hosts=NUM_HOSTS, host_hidden_dim=int(base_config["model"]["host_hidden_dim"]), freeze_encoder=True, bottleneck_hidden_dim=64).to(DEVICE)

def context_source_forward(model, flat):
    z0 = model.encode(flat); raw, context = model.infer_context(z0)
    route = torch.softmax(model.context_bottleneck_router(context) / TAU, dim=-1)
    return model.synthesize(z0, context, route, context_raw=raw)

def train_aligned_context_checkpoint(model_seed):
    torch.manual_seed(model_seed + 1090)
    model = build_context_source_model()
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(params, lr=float(base_config["training"]["learning_rate"]), weight_decay=float(base_config["training"]["weight_decay"]))
    rows = []
    for stage, current_angle in enumerate(CURRICULUM):
        seen = CURRICULUM[:stage+1]
        for epoch in range(CONTEXT_STAGE_EPOCHS):
            for views, y, factors, _ in multi_loader(seen, True, current_indices, True, epoch_seed(model_seed, stage, epoch, 109)):
                batch, angle_count = views.shape[:2]
                y, factors = y.to(DEVICE), factors.to(DEVICE)
                trace = context_source_forward(model, views.reshape(-1,*views.shape[2:]).to(DEVICE))
                logits = trace.logits.reshape(batch, angle_count, -1)
                routes = trace.route.reshape(batch, angle_count, -1)
                contexts = trace.context.reshape(batch, angle_count, -1)
                hosts = trace.host_outputs.reshape(batch, angle_count, NUM_HOSTS, -1)
                current_ce = F.cross_entropy(logits[:,-1], y)
                anchor_ce = F.cross_entropy(logits[:,:-1].reshape(-1,logits.shape[-1]), y[:,None].expand(-1,angle_count-1).reshape(-1)) if angle_count > 1 else logits.sum()*0.0
                loss = current_ce + 0.10*anchor_ce + 0.10*paired_route_geometry_loss_stationary(routes,factors) + 0.10*paired_context_geometry_loss(contexts,factors) + 0.05*context_path_spread_loss(contexts,0.05) + 0.05*cross_source_fiber_alignment_loss(contexts,factors) + 0.05*factor_centroid_geometry_loss(contexts,factors) + 0.05*host_functional_diversity_loss(hosts[:,-1])
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        rows.append({"model_seed":model_seed,"stage":stage,"current_angle":current_angle})
    checkpoint={"encoder":copy.deepcopy(model.encoder.state_dict()),"context_net":copy.deepcopy(model.context_net.state_dict())}
    return model, checkpoint, state_hash(checkpoint["context_net"]), pd.DataFrame(rows)

@torch.no_grad()
def collect_context_bank(source_model, angles, indices):
    contexts=[]; source_model.eval()
    for views,_,_,_ in multi_loader(angles,True,indices,False):
        flat=views.reshape(-1,*views.shape[2:]).to(DEVICE); z0=source_model.encode(flat); _,c=source_model.infer_context(z0); contexts.append(c.cpu())
    return torch.cat(contexts)


@torch.no_grad()
def evaluate_sensory_reconstruction():
    encoder.eval(); sensory_head.eval(); total = correct = 0
    for x, y, _, _ in fixed_loader(0.0, False, sensory_test_indices, False, batch_size=256):
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += int((sensory_head(encoder(x)).argmax(1) == y).sum())
        total += y.numel()
    return correct / max(total, 1)

sensory_accuracy = evaluate_sensory_reconstruction()
minimum_sensory_accuracy = float(RECONSTRUCTION_CFG["minimum_sensory_accuracy"])
assert sensory_accuracy >= minimum_sensory_accuracy, (
    f"Reconstruction acceptance failed: sensory accuracy {sensory_accuracy:.4f} < {minimum_sensory_accuracy:.4f}"
)
reconstruction_acceptance = {
    "sensory_accuracy": sensory_accuracy,
    "minimum_sensory_accuracy": minimum_sensory_accuracy,
    "reference_v112_sensory_accuracy": float(RECONSTRUCTION_CFG["v112_reference_sensory_accuracy"]),
    "exact_hashes_are_gate": bool(RECONSTRUCTION_CFG["exact_hashes_are_gate"]),
    "accepted": True,
}
print("Reconstruction acceptance:", reconstruction_acceptance)


## 5. Construct and freeze one common host bank and cost matrix


In [ ]:
def build_common_bank_model(checkpoint, model_seed):
    torch.manual_seed(model_seed + 1110)
    enc=SmallConvEncoder(latent_dim); enc.load_state_dict(checkpoint["encoder"])
    context_net=nn.Sequential(nn.Linear(latent_dim,64),nn.GELU(),nn.Linear(64,CONTEXT_DIM)); context_net.load_state_dict(checkpoint["context_net"])
    hosts=[ResidualHost(latent_dim,int(base_config["model"]["host_hidden_dim"])) for _ in range(NUM_HOSTS)]
    return FunctionalRoutingMMALS(enc,context_net,UniformRouter(NUM_HOSTS),hosts,nn.LayerNorm(latent_dim),nn.Linear(latent_dim,10),freeze_encoder=True,freeze_context=True).to(DEVICE)

def train_common_host_bank(model_seed, checkpoint):
    model=build_common_bank_model(checkpoint,model_seed)
    params=[p for name,p in model.named_parameters() if p.requires_grad and not name.startswith("router.")]
    optimizer=torch.optim.AdamW(params,lr=float(base_config["training"]["learning_rate"]),weight_decay=float(base_config["training"]["weight_decay"]))
    rows=[]
    for stage,current_angle in enumerate(CURRICULUM):
        seen=CURRICULUM[:stage+1]
        for epoch in range(BANK_STAGE_EPOCHS):
            for views,y,_,_ in multi_loader(seen,True,current_indices,True,epoch_seed(model_seed,stage,epoch,511)):
                batch,angle_count=views.shape[:2]; y=y.to(DEVICE)
                trace=model(views.reshape(-1,*views.shape[2:]).to(DEVICE),temperature=TAU)
                logits=trace.logits.reshape(batch,angle_count,-1); hosts=trace.host_outputs.reshape(batch,angle_count,NUM_HOSTS,-1)
                current_ce=F.cross_entropy(logits[:,-1],y)
                anchor_ce=F.cross_entropy(logits[:,:-1].reshape(-1,logits.shape[-1]),y[:,None].expand(-1,angle_count-1).reshape(-1)) if angle_count>1 else logits.sum()*0.0
                loss=current_ce+0.1*anchor_ce+0.05*host_functional_diversity_loss(hosts[:,-1])
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        rows.append({"model_seed":model_seed,"stage":stage,"current_angle":current_angle})
    state={"hosts":copy.deepcopy(model.hosts.state_dict()),"synthesis_norm":copy.deepcopy(model.synthesis_norm.state_dict()),"classifier":copy.deepcopy(model.classifier.state_dict())}
    flat={**{f"hosts.{k}":v for k,v in state["hosts"].items()},**{f"synthesis_norm.{k}":v for k,v in state["synthesis_norm"].items()},**{f"classifier.{k}":v for k,v in state["classifier"].items()}}
    return model,state,state_hash(flat),pd.DataFrame(rows)

@torch.no_grad()
def build_common_host_cost(model):
    outputs=[]; model.eval()
    for views,_,_,_ in multi_loader(DENSE_EVAL_ANGLES,False,metric_indices,False):
        trace=model(views.reshape(-1,*views.shape[2:]).to(DEVICE),temperature=TAU); outputs.append(trace.host_outputs.cpu())
    return host_output_cost_matrix(torch.cat(outputs),normalize=True)


## 6. Linear-router treatments and original-stage snapshots


In [ ]:
def build_linear_model(checkpoint, bank_state, model_seed):
    torch.manual_seed(model_seed + int(protocol["router"]["initialization_seed_offset"]))
    enc=SmallConvEncoder(latent_dim); enc.load_state_dict(checkpoint["encoder"])
    context_net=nn.Sequential(nn.Linear(latent_dim,64),nn.GELU(),nn.Linear(64,CONTEXT_DIM)); context_net.load_state_dict(checkpoint["context_net"])
    hosts=[ResidualHost(latent_dim,int(base_config["model"]["host_hidden_dim"])) for _ in range(NUM_HOSTS)]
    host_list=nn.ModuleList(hosts); host_list.load_state_dict(bank_state["hosts"])
    norm=nn.LayerNorm(latent_dim); norm.load_state_dict(bank_state["synthesis_norm"])
    classifier=nn.Linear(latent_dim,10); classifier.load_state_dict(bank_state["classifier"])
    model=FunctionalRoutingMMALS(enc,context_net,LinearContextRouter(CONTEXT_DIM,NUM_HOSTS),hosts,norm,classifier,freeze_encoder=True,freeze_context=True).to(DEVICE)
    for module in [model.hosts,model.synthesis_norm,model.classifier]:
        for p in module.parameters(): p.requires_grad_(False)
    return model

@torch.no_grad()
def trace_angle(model, angle, indices, train=False):
    rows=[]; model.eval()
    for x,y,_,source_ids in fixed_loader(angle,train,indices,False,batch_size=128):
        x,y=x.to(DEVICE),y.to(DEVICE); trace=model(x,temperature=TAU)
        for i,sid in enumerate(source_ids):
            rows.append({
                "source_index":int(sid),"angle":float(angle),"label":int(y[i]),
                "route":trace.route[i].cpu().numpy(),"z_mm":trace.z_mm[i].cpu().numpy(),
                "logits":trace.logits[i].cpu().numpy(),"prediction":int(trace.logits[i].argmax()),
            })
    return pd.DataFrame(rows)

def stack(frame,column): return np.stack(frame[column].to_numpy())

@torch.no_grad()
def evaluate_stage(model, model_seed, method, stage):
    rows=[]; model.eval()
    for angle in CURRICULUM:
        total=correct=0; nll_sum=0.0
        for x,y,_,_ in fixed_loader(float(angle),False,safety_eval_indices,False,batch_size=256):
            x,y=x.to(DEVICE),y.to(DEVICE); trace=model(x,temperature=TAU)
            total += y.numel(); correct += int((trace.logits.argmax(1)==y).sum())
            nll_sum += float(F.cross_entropy(trace.logits,y,reduction="sum"))
        rows.append({
            "model_seed":model_seed,"method":method,"stage":stage,"angle":float(angle),
            "accuracy":correct/max(total,1),"nll":nll_sum/max(total,1),
            "is_current":int(float(angle)==float(CURRICULUM[stage])),
            "is_learned":int(CURRICULUM.index(angle)<=stage),
        })
    return rows

def training_snapshot_for_method(full_snapshot, method):
    columns=["source_index","angle","label"]
    spec=METHOD_SPECS[method]
    if float(spec.get("route_anchor") or 0.0)>0 or float(spec.get("functional_transport") or 0.0)>0:
        columns.append("route")
    if float(spec.get("latent_anchor") or 0.0)>0:
        columns.append("z_mm")
    if float(spec.get("output_distillation") or 0.0)>0:
        columns.append("logits")
    return full_snapshot[columns].copy()

def memory_information_budget(method):
    spec=METHOD_SPECS[method]
    fields=["source_index","label","raw_image_access"]
    floats=0
    if float(spec.get("route_anchor") or 0.0)>0 or float(spec.get("functional_transport") or 0.0)>0:
        fields.append("route"); floats += NUM_HOSTS
    if float(spec.get("latent_anchor") or 0.0)>0:
        fields.append("z_mm"); floats += latent_dim
    if float(spec.get("output_distillation") or 0.0)>0:
        fields.append("logits"); floats += 10
    return {
        "method":method,
        "remembered_sources_per_angle":MEMORY_PER_ANGLE,
        "gradient_eligible_fields":json.dumps(fields),
        "auxiliary_target_float_count_per_source":floats,
        "estimated_auxiliary_target_bytes_per_source":4*floats,
        "budget_definition":protocol["experiment"]["memory_budget_definition"],
    }


## 7. Limited-memory training objective

All treatments execute the same current and memory forward passes. M0 computes the memory trace but assigns zero weight to it, preserving matched compute.


In [ ]:
def tensor_from_rows(frame, column, device=DEVICE):
    return torch.tensor(np.stack(frame[column].to_numpy()),dtype=torch.float32,device=device)

def train_memory_treatment(model_seed, method, model, common_cost, checkpoint, bank_state):
    spec=METHOD_SPECS[method]
    params=[p for p in model.router.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=float(base_config["training"]["learning_rate"]),weight_decay=float(base_config["training"]["weight_decay"]))
    stage_rows=[]; snapshots={}; router_states=[]; evaluation_rows=[]; snapshot_events=[]
    for stage,current_angle in enumerate(CURRICULUM):
        current_loader=fixed_loader(current_angle,True,current_indices,True,epoch_seed(model_seed,stage,0,1301),batch_size=SOURCE_BATCH_SIZE)
        old_angles=CURRICULUM[:stage]
        for epoch in range(ROUTER_STAGE_EPOCHS):
            for x,y,_,_ in current_loader:
                x,y=x.to(DEVICE),y.to(DEVICE); current=model(x,temperature=TAU)
                current_ce=F.cross_entropy(current.logits,y)
                replay_ce=current_ce*0.0; route_loss=current_ce*0.0; functional_loss=current_ce*0.0; latent_loss=current_ce*0.0; output_loss=current_ce*0.0
                memory_forward_images=0
                for old_angle in old_angles:
                    memory_frame=snapshots[float(old_angle)]
                    ids=memory_frame.source_index.tolist()
                    mx,my,_,_=next(iter(fixed_loader(old_angle,True,ids,False,batch_size=len(ids))))
                    mx,my=mx.to(DEVICE),my.to(DEVICE); now=model(mx,temperature=TAU)
                    memory_forward_images += my.numel()
                    replay_ce = replay_ce + F.cross_entropy(now.logits,my)
                    if "route" in memory_frame:
                        ref_route=tensor_from_rows(memory_frame,"route")
                        route_loss = route_loss + root_route_anchor_loss(now.route,ref_route)
                        functional_loss = functional_loss + functional_transport_anchor_loss(
                            now.route,ref_route,common_cost.to(DEVICE),
                            epsilon=float(protocol["functional_transport"]["route_sinkhorn_epsilon"]),
                            iterations=int(protocol["functional_transport"]["route_sinkhorn_iterations"]),
                        )
                    if "z_mm" in memory_frame:
                        latent_loss = latent_loss + latent_anchor_loss(now.z_mm,tensor_from_rows(memory_frame,"z_mm"))
                    if "logits" in memory_frame:
                        output_loss = output_loss + distillation_kl(
                            now.logits,tensor_from_rows(memory_frame,"logits"),
                            temperature=float(protocol["training"]["distillation_temperature"]),
                        )
                denom=max(len(old_angles),1)
                loss=(
                    current_ce
                    + float(spec["replay_ce"])*replay_ce/denom
                    + float(spec["route_anchor"])*route_loss/denom
                    + float(spec["functional_transport"] or 0.0)*functional_loss/denom
                    + float(spec["latent_anchor"])*latent_loss/denom
                    + float(spec["output_distillation"])*output_loss/denom
                    + float(protocol["training"]["usage_balance"])*usage_balance_loss(current.route)
                )
                optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
                stage_rows.append({
                    "model_seed":model_seed,"method":method,"stage":stage,"current_angle":current_angle,
                    "loss":float(loss.detach()),"current_ce":float(current_ce.detach()),
                    "replay_ce":float((replay_ce/denom).detach()),"route_anchor":float((route_loss/denom).detach()),
                    "functional_transport":float((functional_loss/denom).detach()),
                    "latent_anchor":float((latent_loss/denom).detach()),"output_distillation":float((output_loss/denom).detach()),
                    "memory_forward_images":memory_forward_images,"current_forward_images":y.numel(),"optimizer_steps":1,
                })
        # Contract: capture immediately after the final optimizer step and before any next-stage operation.
        acquisition_state=copy.deepcopy(model.router.state_dict())
        teacher=build_linear_model(checkpoint,bank_state,model_seed)
        teacher.router.load_state_dict(acquisition_state); teacher.eval()
        full_snapshot=trace_angle(teacher,float(current_angle),memory_indices,train=True)
        snapshots[float(current_angle)] = training_snapshot_for_method(full_snapshot,method)
        router_states.append(acquisition_state)
        snapshot_events.append({
            "model_seed":model_seed,"method":method,"stage":stage,"angle":float(current_angle),
            "event_order":"final_optimizer_step -> deep_copy_router -> frozen_teacher_targets -> stage_evaluation -> next_stage",
            "router_state_sha256":state_hash(acquisition_state),
            "remembered_sources":len(full_snapshot),
            "target_columns":json.dumps(list(snapshots[float(current_angle)].columns)),
        })
        evaluation_rows.extend(evaluate_stage(model,model_seed,method,stage))
    return model,pd.DataFrame(stage_rows),pd.DataFrame(evaluation_rows),snapshots,router_states,pd.DataFrame(snapshot_events)


## 7.1 Mandatory development gradient-scale calibration

Run this section with `G1_PROFILE=development_calibration`. It creates a report proposing `lambda_F` from median unweighted router-gradient norms. Commit the proposed value and report SHA-256 into the YAML before any pilot run. The pilot profile is intentionally blocked while calibration is pending.

In [ ]:
def router_gradient_l2(loss, params, retain_graph=True):
    gradients=torch.autograd.grad(loss,params,retain_graph=retain_graph,allow_unused=True)
    total=torch.zeros((),device=loss.device)
    for gradient in gradients:
        if gradient is not None:
            total=total+gradient.square().sum()
    return float(torch.sqrt(total).detach().cpu())

def run_gradient_scale_calibration(model_seed=0):
    random.seed(model_seed); np.random.seed(model_seed); torch.manual_seed(model_seed)
    source_model,checkpoint,context_hash,_=train_aligned_context_checkpoint(model_seed)
    bank_model,bank_state,bank_hash,_=train_common_host_bank(model_seed,checkpoint)
    common_cost=build_common_host_cost(bank_model)
    model=build_linear_model(checkpoint,bank_state,model_seed)
    params=[p for p in model.router.parameters() if p.requires_grad]
    optimizer=torch.optim.AdamW(params,lr=float(base_config["training"]["learning_rate"]),weight_decay=float(base_config["training"]["weight_decay"]))

    # Acquire the first angle after a current-only stage.
    first_angle=float(CURRICULUM[0])
    for x,y,_,_ in fixed_loader(first_angle,True,current_indices,True,epoch_seed(model_seed,0,0,1701),batch_size=SOURCE_BATCH_SIZE):
        x,y=x.to(DEVICE),y.to(DEVICE); trace=model(x,temperature=TAU)
        loss=F.cross_entropy(trace.logits,y)+float(protocol["training"]["usage_balance"])*usage_balance_loss(trace.route)
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
    acquisition_state=copy.deepcopy(model.router.state_dict())
    teacher=build_linear_model(checkpoint,bank_state,model_seed); teacher.router.load_state_dict(acquisition_state)
    reference=trace_angle(teacher,first_angle,memory_indices,train=True)
    ref_route=tensor_from_rows(reference,"route")
    ids=reference.source_index.tolist()
    mx,my,_,_=next(iter(fixed_loader(first_angle,True,ids,False,batch_size=len(ids))))
    mx,my=mx.to(DEVICE),my.to(DEVICE)

    route_norms=[]; functional_norms=[]; rows=[]
    second_angle=float(CURRICULUM[1])
    probe_batches=int(CALIBRATION_CFG["probe_batches"])
    for probe in range(probe_batches):
        cx,cy,_,_=next(iter(fixed_loader(second_angle,True,current_indices,True,epoch_seed(model_seed,1,probe,1801),batch_size=SOURCE_BATCH_SIZE)))
        cx,cy=cx.to(DEVICE),cy.to(DEVICE)
        current=model(cx,temperature=TAU); replay=model(mx,temperature=TAU)
        drift_loss=(F.cross_entropy(current.logits,cy)+0.5*F.cross_entropy(replay.logits,my)+float(protocol["training"]["usage_balance"])*usage_balance_loss(current.route))
        optimizer.zero_grad(set_to_none=True); drift_loss.backward(); optimizer.step()

        now=model(mx,temperature=TAU)
        route_loss=root_route_anchor_loss(now.route,ref_route)
        functional_loss=functional_transport_anchor_loss(
            now.route,ref_route,common_cost.to(DEVICE),
            epsilon=float(protocol["functional_transport"]["route_sinkhorn_epsilon"]),
            iterations=int(protocol["functional_transport"]["route_sinkhorn_iterations"]),
        )
        route_norm=router_gradient_l2(route_loss,params,retain_graph=True)
        functional_norm=router_gradient_l2(functional_loss,params,retain_graph=False)
        route_norms.append(route_norm); functional_norms.append(functional_norm)
        rows.append({"probe_batch":probe,"route_anchor_grad_l2":route_norm,"functional_transport_grad_l2":functional_norm})

    median_route=float(np.median(route_norms)); median_functional=float(np.median(functional_norms))
    if median_route<=0 or median_functional<=0:
        raise RuntimeError(f"Invalid calibration norms: route={median_route}, functional={median_functional}")
    lambda_r=float(CALIBRATION_CFG["route_anchor_weight"])
    proposed=float(np.clip(
        lambda_r*median_route/median_functional,
        float(CALIBRATION_CFG["coefficient_clip_min"]),
        float(CALIBRATION_CFG["coefficient_clip_max"]),
    ))
    report={
        "version":NOTEBOOK_VERSION,"build_revision":BUILD_REVISION,"calibration_seed":model_seed,
        "probe_batches":probe_batches,"median_route_anchor_grad_l2":median_route,
        "median_functional_transport_grad_l2":median_functional,"route_anchor_weight":lambda_r,
        "proposed_functional_transport_weight":proposed,"clip_min":float(CALIBRATION_CFG["coefficient_clip_min"]),
        "clip_max":float(CALIBRATION_CFG["coefficient_clip_max"]),"context_hash":context_hash,
        "common_bank_hash":bank_hash,"sensory_accuracy":sensory_accuracy,
        "decision":"commit proposed weight and report SHA-256 before pilot; no adaptive rescaling during pilot",
    }
    calibration_root=Path("results/v1_1_3_calibration"); calibration_root.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(rows).to_csv(calibration_root/"gradient_norm_probes.csv",index=False)
    report_path=calibration_root/"gradient_scale_calibration.json"
    report_path.write_text(json.dumps(report,indent=2),encoding="utf-8")
    report_sha=hashlib.sha256(report_path.read_bytes()).hexdigest()
    (calibration_root/"gradient_scale_calibration.sha256").write_text(
        f"{report_sha}  {report_path.name}\n", encoding="utf-8"
    )
    print(json.dumps(report,indent=2))
    print("YAML lock values:")
    print("  status: locked")
    print("  pilot_lock: true")
    print(f"  calibrated_functional_transport_weight: {proposed:.12g}")
    print(f"  calibration_report_sha256: {report_sha}")
    return report,pd.DataFrame(rows)

if CALIBRATION_ONLY:
    calibration_report,calibration_probe_df=run_gradient_scale_calibration(int(CALIBRATION_CFG["calibration_seed"]))
else:
    print("Calibration lock accepted for execution:", CALIBRATION_CFG["calibration_report_sha256"])


## 8. Source-disjoint memory drift and path diagnostics


In [ ]:
@torch.no_grad()
def evaluate_memory_drift(model_builder, checkpoint, bank_state, model_seed, method, final_model, router_states, common_cost):
    rows, path_rows, compiled_rows, distributional_rows = [], [], [], []
    for learned_stage, angle in enumerate(CURRICULUM):
        original = model_builder(checkpoint, bank_state, model_seed)
        original.router.load_state_dict(router_states[learned_stage])
        original.eval(); final_model.eval()
        ref = trace_angle(original, float(angle), memory_eval_indices, train=False)
        cur = trace_angle(final_model, float(angle), memory_eval_indices, train=False)
        merged = ref.merge(cur, on=["source_index", "angle", "label"], suffixes=("_ref", "_cur"))
        ref_r = tensor_from_rows(merged, "route_ref", device="cpu")
        cur_r = tensor_from_rows(merged, "route_cur", device="cpu")
        ref_z = tensor_from_rows(merged, "z_mm_ref", device="cpu")
        cur_z = tensor_from_rows(merged, "z_mm_cur", device="cpu")
        ref_l = tensor_from_rows(merged, "logits_ref", device="cpu")
        cur_l = tensor_from_rows(merged, "logits_cur", device="cpu")
        functional = entropic_transport_cost(
            cur_r,
            ref_r,
            common_cost.float(),
            epsilon=float(protocol["functional_transport"]["route_sinkhorn_epsilon"]),
            iterations=int(protocol["functional_transport"]["route_sinkhorn_iterations"]),
        )
        route = 0.5 * torch.square(torch.sqrt(cur_r.clamp_min(1e-12)) - torch.sqrt(ref_r.clamp_min(1e-12))).sum(-1)
        latent = torch.square(cur_z - ref_z).mean(-1)
        teacher = torch.softmax(ref_l / 2.0, -1)
        output_kl = F.kl_div(torch.log_softmax(cur_l / 2.0, -1), teacher, reduction="none").sum(-1) * 4.0
        identity = (cur_l.argmax(-1) == ref_l.argmax(-1)).float()
        for i, sid in enumerate(merged.source_index):
            rows.append({
                "model_seed": model_seed,
                "method": method,
                "angle": float(angle),
                "source_index": int(sid),
                "route_drift": float(route[i]),
                "functional_drift": float(functional[i]),
                "latent_drift": float(latent[i]),
                "output_kl": float(output_kl[i]),
                "identity_preserved": float(identity[i]),
            })
        limit = min(DISTRIBUTIONAL_LIMIT, len(ref_r), len(cur_r))
        distributional_rows.append({
            "model_seed": model_seed,
            "method": method,
            "angle": float(angle),
            "sources": limit,
            "distributional_functional_transport": float(distributional_functional_transport(
                ref_r[:limit],
                cur_r[:limit],
                common_cost.float(),
                route_epsilon=float(protocol["functional_transport"]["route_sinkhorn_epsilon"]),
                route_iterations=int(protocol["functional_transport"]["route_sinkhorn_iterations"]),
                cloud_epsilon=float(protocol["functional_transport"]["cloud_sinkhorn_epsilon"]),
                cloud_iterations=int(protocol["functional_transport"]["cloud_sinkhorn_iterations"]),
            )),
        })
        stage_routes = []
        for state in router_states[learned_stage:]:
            staged = model_builder(checkpoint, bank_state, model_seed)
            staged.router.load_state_dict(state)
            stage_routes.append(torch.tensor(stack(trace_angle(staged, float(angle), path_eval_indices, train=False), "route"), dtype=torch.float32))
        if len(stage_routes) >= 2:
            ps = functional_path_statistics(
                stage_routes,
                common_cost.float(),
                epsilon=float(protocol["functional_transport"]["route_sinkhorn_epsilon"]),
                iterations=int(protocol["functional_transport"]["route_sinkhorn_iterations"]),
            )
            path_rows.append({
                "model_seed": model_seed,
                "method": method,
                "angle": float(angle),
                "cumulative_length": ps.cumulative_length,
                "endpoint_distance": ps.endpoint_distance,
                "path_to_endpoint_cost_ratio": ps.path_to_endpoint_cost_ratio,
                "path_excess": ps.path_to_endpoint_cost_ratio,
                "steps": ps.steps,
            })
        fit_routes = torch.tensor(stack(trace_angle(original, float(angle), compiled_fit_indices, train=False), "route"), dtype=torch.float32)
        eval_routes = torch.tensor(stack(trace_angle(original, float(angle), compiled_eval_indices, train=False), "route"), dtype=torch.float32)
        compiled = compile_root_gaussian(
            fit_routes,
            shrinkage=float(protocol["compiled_memory_diagnostic"]["root_gaussian_shrinkage"]),
            eps=float(protocol["compiled_memory_diagnostic"]["root_gaussian_epsilon"]),
        )
        isotropic = isotropic_root_gaussian(compiled)
        compiled_rows.append({
            "model_seed": model_seed,
            "method": method,
            "angle": float(angle),
            "covariance_nll": float(root_gaussian_nll(eval_routes, compiled)),
            "isotropic_nll": float(root_gaussian_nll(eval_routes, isotropic)),
        })
    return pd.DataFrame(rows), pd.DataFrame(path_rows), pd.DataFrame(compiled_rows), pd.DataFrame(distributional_rows)


## 9. Per-seed execution


In [ ]:
def max_state_delta(module, reference_state):
    current = module.state_dict()
    return max(float((current[name].detach().cpu() - reference_state[name].detach().cpu()).abs().max()) for name in reference_state)


def run_seed(model_seed):
    print(f"\n===== MODEL SEED {model_seed} =====")
    random.seed(model_seed); np.random.seed(model_seed); torch.manual_seed(model_seed)
    source_model, checkpoint, context_hash, context_stage = train_aligned_context_checkpoint(model_seed)
    context_bank = collect_context_bank(source_model, TRAIN_ANGLES, current_indices)
    bank_model, bank_state, bank_hash, bank_stage = train_common_host_bank(model_seed, checkpoint)
    common_cost = build_common_host_cost(bank_model)
    stages=[]; evaluations=[]; drifts=[]; paths=[]; compiled=[]; distributions=[]; summaries=[]; preservation=[]; snapshot_events=[]
    base_model = build_linear_model(checkpoint, bank_state, model_seed)
    initial_router = copy.deepcopy(base_model.router.state_dict())
    for method in METHODS:
        model = build_linear_model(checkpoint, bank_state, model_seed)
        model.router.load_state_dict(initial_router)
        model, stage, evaluation, snapshots, router_states, events = train_memory_treatment(model_seed, method, model, common_cost, checkpoint, bank_state)
        stages.append(stage); evaluations.append(evaluation); snapshot_events.append(events)
        d, p, c, dist = evaluate_memory_drift(build_linear_model, checkpoint, bank_state, model_seed, method, model, router_states, common_cost)
        drifts.append(d); paths.append(p); compiled.append(c); distributions.append(dist)
        preservation.append({
            "model_seed": model_seed,
            "method": method,
            "max_encoder_delta": max_state_delta(model.encoder, checkpoint["encoder"]),
            "max_context_delta": max_state_delta(model.context_net, checkpoint["context_net"]),
            "max_hosts_delta": max_state_delta(model.hosts, bank_state["hosts"]),
            "max_norm_delta": max_state_delta(model.synthesis_norm, bank_state["synthesis_norm"]),
            "max_classifier_delta": max_state_delta(model.classifier, bank_state["classifier"]),
        })
        final_stage=int(evaluation.stage.max())
        final_eval=evaluation[(evaluation.stage==final_stage)&(evaluation.is_learned==1)]
        forgetting_rows=[]
        for angle in CURRICULUM:
            learned_stage=CURRICULUM.index(angle)
            history=evaluation[(evaluation.angle==angle)&(evaluation.stage>=learned_stage)]
            best=float(history.accuracy.max())
            final=float(history[history.stage==final_stage].accuracy.iloc[0])
            forgetting_rows.append(best-final)
        summaries.append({
            "model_seed":model_seed,"method":method,
            "final_trained_accuracy":float(final_eval.accuracy.mean()),
            "mean_forgetting":float(np.mean(forgetting_rows)),
            "mean_new_angle_accuracy":float(evaluation[evaluation.is_current==1].accuracy.mean()),
            "router_parameters":parameter_count(model.router),
        })
    return {
        "context_hash":context_hash,"bank_hash":bank_hash,"common_cost":common_cost,
        "stage":pd.concat(stages,ignore_index=True),"evaluation":pd.concat(evaluations,ignore_index=True),
        "drift":pd.concat(drifts,ignore_index=True),"paths":pd.concat(paths,ignore_index=True),
        "compiled":pd.concat(compiled,ignore_index=True),"distributional":pd.concat(distributions,ignore_index=True),
        "summary":pd.DataFrame(summaries),"preservation":pd.DataFrame(preservation),
        "snapshot_events":pd.concat(snapshot_events,ignore_index=True),
    }

if CALIBRATION_ONLY:
    all_runs=[]
    print("Calibration-only profile complete. Lock the YAML before running development/pilot.")
else:
    all_runs=[(seed,run_seed(seed)) for seed in MODEL_SEEDS]
    print("Completed seeds:",[seed for seed,_ in all_runs])


## 10. Aggregation and preregistered gates


In [ ]:
if CALIBRATION_ONLY:
    print("Aggregation skipped in calibration-only profile.")
else:
    def concat_key(key): return pd.concat([run[key] for _, run in all_runs], ignore_index=True)
    stage_df = concat_key("stage")
    snapshot_event_df = concat_key("snapshot_events")
    memory_budget_df = pd.DataFrame([memory_information_budget(method) for method in METHODS])
    evaluation_df = concat_key("evaluation")
    drift_df = concat_key("drift")
    path_df = concat_key("paths")
    compiled_df = concat_key("compiled")
    distributional_df = concat_key("distributional")
    summary_df = concat_key("summary")
    preservation_df = concat_key("preservation")


    def seed_ci(values, confidence=.95):
        values = np.asarray(values, dtype=float); n = len(values)
        mean = float(values.mean()); sd = float(values.std(ddof=1)) if n > 1 else 0.0
        if n < 2: return mean, mean, mean, sd, n
        half = float(stats.t.ppf((1 + confidence) / 2, df=n - 1) * sd / math.sqrt(n))
        return mean, mean - half, mean + half, sd, n

    seed_drift = drift_df.groupby(["model_seed", "method"], as_index=False).agg(
        functional_drift=("functional_drift", "mean"),
        route_drift=("route_drift", "mean"),
        latent_drift=("latent_drift", "mean"),
        output_kl=("output_kl", "mean"),
        identity=("identity_preserved", "mean"),
    )
    seed_paths = path_df.groupby(["model_seed", "method"], as_index=False).agg(
        path_to_endpoint_cost_ratio=("path_to_endpoint_cost_ratio", "mean"),
        cumulative_length=("cumulative_length", "mean"),
        endpoint_distance=("endpoint_distance", "mean"),
    )
    seed_compiled = compiled_df.groupby(["model_seed", "method"], as_index=False).agg(
        covariance_nll=("covariance_nll", "mean"),
        isotropic_nll=("isotropic_nll", "mean"),
    )
    seed_distributional = distributional_df.groupby(["model_seed", "method"], as_index=False).agg(
        distributional_functional_transport=("distributional_functional_transport", "mean")
    )

    contrast_specs = [
        ("m3_functional_transport", "m2_route_anchor", "functional_drift", seed_drift, "less"),
        ("m4_joint_functional_memory", "m1_replay_ce", "mean_forgetting", summary_df, "less"),
        ("m3_functional_transport", "m2_route_anchor", "mean_forgetting", summary_df, "less"),
        ("m4_joint_functional_memory", "m1_replay_ce", "path_to_endpoint_cost_ratio", seed_paths, "less"),
        ("m4_joint_functional_memory", "m1_replay_ce", "mean_new_angle_accuracy", summary_df, "equivalent"),
    ]
    contrasts = []
    for treatment, control, metric, table, direction in contrast_specs:
        left = table[table.method == treatment].set_index("model_seed")[metric]
        right = table[table.method == control].set_index("model_seed")[metric]
        delta = left - right
        mean, low, high, sd, n = seed_ci(delta)
        favorable = np.abs(delta) <= 0.01 if direction == "equivalent" else delta < 0
        contrasts.append({
            "contrast": f"{treatment}_vs_{control}",
            "metric": metric,
            "mean_seed_effect": mean,
            "seed_ci_low": low,
            "seed_ci_high": high,
            "seed_sd": sd,
            "seed_count": n,
            "favorable_seed_fraction": float(np.mean(favorable)),
        })
    aggregate_effects_df = pd.DataFrame(contrasts)

    def effect(contrast, metric):
        return aggregate_effects_df[(aggregate_effects_df.contrast == contrast) & (aggregate_effects_df.metric == metric)].iloc[0]

    m3 = effect("m3_functional_transport_vs_m2_route_anchor", "functional_drift")
    m4_forgetting = effect("m4_joint_functional_memory_vs_m1_replay_ce", "mean_forgetting")
    m3_forgetting = effect("m3_functional_transport_vs_m2_route_anchor", "mean_forgetting")
    m4_path = effect("m4_joint_functional_memory_vs_m1_replay_ce", "path_to_endpoint_cost_ratio")
    m4_new = effect("m4_joint_functional_memory_vs_m1_replay_ce", "mean_new_angle_accuracy")
    identity = seed_drift[seed_drift.method == "m4_joint_functional_memory"].identity
    compiled_delta = (
        seed_compiled[seed_compiled.method == "m4_joint_functional_memory"].set_index("model_seed").covariance_nll
        - seed_compiled[seed_compiled.method == "m4_joint_functional_memory"].set_index("model_seed").isotropic_nll
    )
    compiled_mean, compiled_low, compiled_high, _, _ = seed_ci(compiled_delta)
    compute = stage_df.groupby(["model_seed", "method"], as_index=False).agg(
        current_forward_images=("current_forward_images", "sum"),
        memory_forward_images=("memory_forward_images", "sum"),
        optimizer_steps=("optimizer_steps", "sum"),
    )
    matched_compute = all(block[["current_forward_images", "memory_forward_images", "optimizer_steps"]].nunique().max() == 1 for _, block in compute.groupby("model_seed"))
    frozen_ok = bool(preservation_df[["max_encoder_delta", "max_context_delta", "max_hosts_delta", "max_norm_delta", "max_classifier_delta"]].to_numpy().max() <= 1e-6)

    candidate_gates = pd.DataFrame([
        {"gate": "C0_frozen_ecosystem", "passed": frozen_ok, "status": "protocol_integrity"},
        {"gate": "C0_matched_compute", "passed": bool(matched_compute), "status": "protocol_integrity"},
        {"gate": "C0_reconstruction_acceptance", "passed": bool(reconstruction_acceptance["accepted"]), "status": "protocol_integrity"},
        {"gate": "C0_remembered_source_budget", "passed": bool(len(memory_indices) == MEMORY_PER_ANGLE and set(memory_indices).isdisjoint(current_indices)), "status": "protocol_integrity"},
        {"gate": "C0_snapshot_ordering", "passed": bool(snapshot_event_df.event_order.str.contains("final_optimizer_step -> deep_copy_router").all()), "status": "protocol_integrity"},
        {"gate": "C0_gradient_calibration_locked", "passed": bool(CALIBRATION_CFG.get("status") == "locked" and CALIBRATION_CFG.get("pilot_lock") is True), "status": "protocol_integrity"},
        {"gate": "C5_functional_transport_beats_route_anchor", "passed": bool(m3.seed_ci_high < 0 and m3.favorable_seed_fraction >= 0.8), "status": "candidate_only"},
        {"gate": "C5_functional_transport_forgetting_alignment", "passed": bool(m3_forgetting.seed_ci_high < 0 and m3_forgetting.favorable_seed_fraction >= 0.8), "status": "secondary_candidate_only"},
        {"gate": "C5_joint_memory_reduces_forgetting", "passed": bool(m4_forgetting.seed_ci_high < 0 and m4_forgetting.favorable_seed_fraction >= 0.8), "status": "candidate_only"},
        {"gate": "C5_path_efficiency", "passed": bool(m4_path.seed_ci_high < 0 and m4_path.favorable_seed_fraction >= 0.8), "status": "candidate_only"},
        {"gate": "C5_identity_preservation", "passed": bool((identity >= 0.95).mean() >= 0.8), "status": "candidate_only"},
        {"gate": "C6_new_angle_non_degradation", "passed": bool(m4_new.favorable_seed_fraction >= 0.8), "status": "equivalence_not_superiority"},
        {"gate": "C5_compiled_covariance_value", "passed": bool(compiled_high < 0 and float((compiled_delta < 0).mean()) >= 0.8), "status": "diagnostic_candidate_only"},
        {"gate": "C3_mature_host_specialization", "passed": False, "status": "not_tested_hosts_frozen"},
        {"gate": "positive_backward_transfer", "passed": False, "status": "not_preregistered_as_primary"},
    ])
    candidate_gates


## 11. Immutable evidence export


In [ ]:
if CALIBRATION_ONLY:
    print("Evidence export skipped; calibration report is in results/v1_1_3_calibration.")
else:
    try:
        git_sha=subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip()
    except Exception:
        git_sha="unknown"
    root_out=Path("results/v1_1_3"); root_out.mkdir(parents=True,exist_ok=True)
    frames={
        "training_stage_logs.csv":stage_df,"staged_evaluation.csv":evaluation_df,
        "memory_drift_raw.csv":drift_df,"memory_path_summary.csv":path_df,
        "distributional_transport.csv":distributional_df,"compiled_memory_diagnostics.csv":compiled_df,
        "per_seed_method_summary.csv":summary_df,"preservation.csv":preservation_df,
        "snapshot_event_log.csv":snapshot_event_df,"remembered_source_budget_summary.csv":memory_budget_df,
        "compute_summary.csv":compute,"aggregate_seed_effects.csv":aggregate_effects_df,
        "gate_summary.csv":candidate_gates,"sensory_history.csv":pd.DataFrame(sensory_history),
        "reconstruction_acceptance.csv":pd.DataFrame([reconstruction_acceptance]),
    }
    for name,frame in frames.items(): frame.to_csv(root_out/name,index=False)
    claim_manifest={
        "schema_version":"1.2","version":NOTEBOOK_VERSION,
        "execution_status":f"{RUN_PROFILE}_completed",
        "build_revision":BUILD_REVISION,
        "primary_treatments":["m3_functional_transport","m4_joint_functional_memory"],
        "budget_definition":protocol["experiment"]["memory_budget_definition"],
        "eligible_if_gates_pass":["functional transport beats route anchoring","joint functional memory beats replay","identity preservation","operational non-degradation"],
        "diagnostic_only":["M3-vs-M2 forgetting alignment","path-to-endpoint regularized cost ratio","compiled covariance value"],
        "non_claims":protocol["non_claims"],
    }
    (root_out/"claim_manifest.json").write_text(json.dumps(claim_manifest,indent=2),encoding="utf-8")
    run_manifest={
        "notebook_version":NOTEBOOK_VERSION,"build_revision":BUILD_REVISION,
        "package_version":geometry_mmalls.__version__,"git_sha":git_sha,"run_profile":RUN_PROFILE,
        "model_seeds":MODEL_SEEDS,"memory_split_manifest":memory_split_manifest,
        "memory_treatments":METHOD_SPECS,"budget_definition":protocol["experiment"]["memory_budget_definition"],
        "gradient_scale_calibration":CALIBRATION_CFG,"reconstruction_acceptance":reconstruction_acceptance,
        "frozen_ecosystem":True,"router_architecture":"linear_context_router",
        "source_v112_results":"results/v1_1_2",
        "warning":"v1.1.3 is the final fully frozen-host experiment; claims require independent verification.",
        "timestamp":time.time(),
    }
    (root_out/"run_manifest.json").write_text(json.dumps(run_manifest,indent=2),encoding="utf-8")
    (root_out/"environment.txt").write_text("\n".join([f"python={sys.version}",f"platform={platform.platform()}",f"torch={torch.__version__}",f"device={DEVICE}"]),encoding="utf-8")
    print("Exported v1.1.3 evidence to",root_out.resolve())


## 12. Interpretation discipline and frozen-regime stopping rule

A successful pilot would show only that functional transport preserves old route-function behavior better than nominal anchoring, or that joint functional memory reduces forgetting beyond replay, under matched remembered-source access and matched compute.

The path quantity is a **path-to-endpoint regularized cost ratio**, not a geodesic excess; values below one are possible because raw entropic OT is not guaranteed to be a metric.

v1.1.3 is the final planned fully frozen-host experiment. Whether the result is positive, negative, or partial, the next scientific regime must test controlled host adaptation rather than another router, distance, or frozen-bank memory variant.

The pilot does not establish mature host specialization, positive backward transfer, operational superiority, byte efficiency, transport optimality, a learned Riemannian manifold, RL control, or quantum advantage.


## 13. Targeted evidence package and PDF export


In [ ]:
from pathlib import Path
import datetime, shutil
notebook_title="Geometry_MMALS_G1_FunctionalMemoryTransport_v1_1_3_C0_R2"
timestamp=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
package_root=Path("/tmp")/f"{notebook_title}_{timestamp}"
if package_root.exists(): shutil.rmtree(package_root)
package_root.mkdir(parents=True)
if CALIBRATION_ONLY:
    shutil.copytree(Path("results/v1_1_3_calibration"),package_root/"results"/"v1_1_3_calibration")
else:
    shutil.copytree(Path("results/v1_1_3"),package_root/"results"/"v1_1_3")
for source in [Path("configs/rotated_mnist_g1_v113.yaml"),Path("docs/specs/Geometry_MMALS_G1_v1_1_3_Functional_Memory_Transport_Specification.md")]:
    if source.exists(): shutil.copy2(source,package_root/source.name)
archive=shutil.make_archive(str(package_root),"zip",package_root)
destination=Path("/content/drive/MyDrive/MMALS")/Path(archive).name
shutil.copy2(archive,destination)
print("Saved",destination)


In [ ]:
# Clean PDF export through Chromium avoids LaTeX failures on long notebook outputs.
!pip -q install "nbconvert[webpdf]"
!jupyter nbconvert --to webpdf --allow-chromium-download --output-dir="/content" "/content/drive/MyDrive/Colab Notebooks/Geometry_MMALS_G1_FunctionalMemoryTransport_v1_1_3_C0_R2.ipynb"
